In [3]:
import polars as pl
import pandas as pd
import pyterrier as pt
import os

if not pt.started():
    pt.init()

/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_17305/1012693727.py:6: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


let us do one query test

In [16]:
INDEX_PATH = os.path.abspath("../indexes/terrier_index_dicty_22.01.26")

# 1) load existing index from disk
index_ref = pt.IndexRef.of(INDEX_PATH)
index = pt.IndexFactory.of(index_ref)

# 2) BM25 retriever
br = pt.BatchRetrieve(index, wmodel="BM25")


/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_17305/308921936.py:8: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  br = pt.BatchRetrieve(index, wmodel="BM25")


In [17]:
# 3) pick one query from gold (change row index if you want)
gold = pl.read_parquet("../output/cleaned/golden_grouped.parquet")

row = gold.select(["group_claim_id", "query", "docs"]).row(0, named=True)


In [18]:
qid = str(row["group_claim_id"])
query = row["query"]
# positives = pmids from gold docs
docs = row["docs"]  # list of structs
pos_pmids = sorted({str(d.get("pmid")) for d in docs if d.get("pmid") not in (None, "", "NA")})
pos_set = set(pos_pmids)

In [21]:
print("qid:", qid)
print("query:", query)
print("n_pos_pmids:", len(pos_pmids))
print("pos_pmids (first 20):", pos_pmids[:20])


qid: 1
query: A basic region in the tail is predicted to bind to acidic phospholipids in the plasma membrane.
n_pos_pmids: 1
pos_pmids (first 20): ['24747353']


In [22]:
# 4) retrieve top K
K = 1000
qdf = pd.DataFrame([{"qid": qid, "query": query}])
res = br.transform(qdf).head(K)
res["docno"] = res["docno"].astype(str)

# 5) check where positives appear
hits = res[res["docno"].isin(pos_set)].copy().sort_values("rank")
missed = sorted(pos_set - set(res["docno"]))

print("\n--- Positive hits in top", K, "---")
print("n_hits:", len(hits), "/", len(pos_pmids))
print("best_rank:", (int(hits["rank"].min()) if len(hits) else None))
print("missed_in_topK:", len(missed))
print("missed pmids (first 20):", missed[:20])

display(hits[["qid","docno","rank","score"]].head(50))

for k in [10, 20, 50, 100, 200, 500, 1000]:
    topk = set(res[res["rank"] < k]["docno"])
    rec = len(topk & pos_set) / (len(pos_set) if len(pos_set) else 1)
    print(f"recall@{k}: {rec:.3f}")


--- Positive hits in top 1000 ---
n_hits: 1 / 1
best_rank: 0
missed_in_topK: 0
missed pmids (first 20): []


,qid,docno,rank,score
0,1,24747353,0,47.925801


recall@10: 1.000
recall@20: 1.000
recall@50: 1.000
recall@100: 1.000
recall@200: 1.000
recall@500: 1.000
recall@1000: 1.000


In [24]:

gid = gold.select("group_claim_id").row(0)[0]   # scalar int
# or: gid = gold.get_column("group_claim_id")[0]

pos_docs = (
    gold
    .filter(pl.col("group_claim_id") == gid)
    .select(["group_claim_id", "query", "docs"])
    .explode("docs")
    .with_columns([
        pl.col("docs").struct.field("pmid").alias("pmid"),
        pl.col("docs").struct.field("publication_id").alias("publication_id"),
        pl.col("docs").struct.field("title").alias("title"),
        pl.col("docs").struct.field("abstract_clean").alias("abstract_clean"),
        pl.col("docs").struct.field("year").alias("year"),
    ])
    .drop("docs")
)

with pl.Config(fmt_str_lengths=10_000, tbl_rows=200, tbl_cols=20):
    display(pos_docs)

group_claim_id,query,pmid,publication_id,title,abstract_clean,year
i64,str,str,i64,str,str,i32
1,"""A basic region in the tail is predicted to bind to acidic phospholipids in the plasma membrane.""","""24747353""",13954,"""The association of myosin IB with actin waves in dictyostelium requires both the plasma membrane-binding site and actin-binding region in the myosin tail.""","""F-actin structures and their distribution are important determinants of the dynamic shapes and functions of eukaryotic cells. Actin waves are F-actin formations that move along the ventral cell membrane driven by actin polymerization. Dictyostelium myosin IB is associated with actin waves but its role in the wave is unknown. Myosin IB is a monomeric, non-filamentous myosin with a globular head that binds to F-actin and has motor activity, and a non-helical tail comprising a basic region, a glycine-proline-glutamine-rich region and an SH3-domain. The basic region binds to acidic phospholipids in the plasma membrane through a short basic-hydrophobic site and the Gly-Pro-Gln region binds F-actin. In the current work we found that both the basic-hydrophobic site in the basic region and the Gly-Pro-Gln region of the tail are required for the association of myosin IB with actin waves. This is the first evidence that the Gly-Pro-Gln region is required for localization of myosin IB to a specific actin structure in situ. The head is not required for myosin IB association with actin waves but binding of the head to F-actin strengthens the association of myosin IB with waves and stabilizes waves. Neither the SH3-domain nor motor activity is required for association of myosin IB with actin waves. We conclude that myosin IB contributes to anchoring actin waves to the plasma membranes by binding of the basic-hydrophobic site to acidic phospholipids in the plasma membrane and binding of the Gly-Pro-Gln region to F-actin in the wave.""",2014
